### Resources for Reinforcement Learning
- https://www.google.com/books/edition/Reinforcement_Learning_second_edition/uWV0DwAAQBAJ?hl=en&gbpv=1&pg=PA1&printsec=frontcover
- https://www.youtube.com/playlist?list=PLzuuYNsE1EZAXYR4FJ75jcJseBmo4KQ9-

## 1. Import Dependencies ##

* Stable_baselines docs: https://stable-baselines3.readthedocs.io/en/master/guide/install.html

In [1]:
# !pip install stable_baselines3[extra]

In [2]:
# !pip install gym
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [3]:
import os
import pygame
import gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


## 2. Load Environments ##

* Gym Environment: https://www.gymlibrary.dev/
* https://github.com/openai/gym/tree/master/gym/envs
* Reinforcement Learning Course: https://github.com/nicknochnack/ReinforcementLearningCourse
* Model based RL vs. Model Free RL: https://spinningup.openai.com/en/latest/
* GPU usage by Pytorch: https://pytorch.org/get-started/locally

In [9]:
env_name = "CartPole-v0"
env = gym.make("CartPole-v1", render_mode="human")

In [10]:
episodes = 5
for episode in range(1, episodes + 1):
    state, _ = env.reset()
    done = False
    score = 0
    
    while not done:
        env.render()  # This shows the environment window if using render_mode="human"  # render func allows us to view the the graphical representation of the env.
        action = env.action_space.sample()
        state, reward, done, _, info = env.step(action)
        score += reward
    print("Episodes:{} score:{}".format(episode, score))
env.close()

Episodes:1 score:53.0
Episodes:2 score:12.0
Episodes:3 score:18.0
Episodes:4 score:27.0
Episodes:5 score:16.0


In [11]:
env.observation_space  #0.cart position, 1.cart velocity, 2.pole angle, 3.pole angular velocity

Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)

In [12]:
env.action_space  #0. push cart to the left, 1.push cart to the right

Discrete(2)

# 3. Training RL Model

In [13]:
log_path = os.path.join("Training", "Logs")  #Make your Dirs first

In [14]:
log_path

'Training\\Logs'

In [15]:
# ImportError: Missing shimmy installation. You provided an OpenAI Gym environment. Stable-Baselines3 (SB3) has transitioned to using Gymnasium internally. 
# In order to use OpenAI Gym environments with SB3, you need to install shimmy (`pip install 'shimmy>=2.0'`).
# !pip install shimmy>=2.0

In [81]:
# Instantiate an Algorithm; specifically an agent
env = gym.make("CartPole-v1")
env = DummyVecEnv([lambda: env])
model = PPO("MlpPolicy", env, verbose = 1, tensorboard_log = log_path)

Using cpu device


In [82]:
help(PPO)

Help on class PPO in module stable_baselines3.ppo.ppo:

class PPO(stable_baselines3.common.on_policy_algorithm.OnPolicyAlgorithm)
 |  PPO(policy: Union[str, type[stable_baselines3.common.policies.ActorCriticPolicy]], env: Union[gymnasium.core.Env, ForwardRef('VecEnv'), str], learning_rate: Union[float, Callable[[float], float]] = 0.0003, n_steps: int = 2048, batch_size: int = 64, n_epochs: int = 10, gamma: float = 0.99, gae_lambda: float = 0.95, clip_range: Union[float, Callable[[float], float]] = 0.2, clip_range_vf: Union[NoneType, float, Callable[[float], float]] = None, normalize_advantage: bool = True, ent_coef: float = 0.0, vf_coef: float = 0.5, max_grad_norm: float = 0.5, use_sde: bool = False, sde_sample_freq: int = -1, rollout_buffer_class: Optional[type[stable_baselines3.common.buffers.RolloutBuffer]] = None, rollout_buffer_kwargs: Optional[dict[str, Any]] = None, target_kl: Optional[float] = None, stats_window_size: int = 100, tensorboard_log: Optional[str] = None, policy_kwa

In [83]:
model.learn(total_timesteps = 10000)

Logging to Training\Logs\PPO_9
-----------------------------
| time/              |      |
|    fps             | 1870 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1302        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008209214 |
|    clip_fraction        | 0.0915      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.686      |
|    explained_variance   | 0.0013      |
|    learning_rate        | 0.0003      |
|    loss                 | 7.35        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0145     |
|    value_loss           | 52.9        |
-----------------------------------------
---

In [84]:
PPO_Path = os.path.join("Training", "Saved Models", "PPO_Model_Cartpole2")

In [85]:
model.save(PPO_Path)

In [86]:
del model

In [87]:
PPO_Path

'Training\\Saved Models\\PPO_Model_Cartpole2'

In [88]:
model = PPO.load(PPO_Path, env= env)

In [89]:
model.learn(total_timesteps= 1000)

Logging to Training\Logs\PPO_10
-----------------------------
| time/              |      |
|    fps             | 1860 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------


In [90]:
# from the above for PPO algorithm we dont get the rollout training metrics like: "ep_len_mean" and "ep_rew_mean". 
# This is very much dependent up on the algorithm.
# ep_len_mean = On avg how long a particular episode lasted before done.
# ep_rew_mean = the avg reward that the agent accumulated per episode.

# 4. Evaluation

In [91]:
evaluate_policy(model, env, n_eval_episodes=10, render=True)

c:\Users\nmokkapa\AppData\Local\anaconda3\Lib\site-packages\stable_baselines3\common\evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
c:\Users\nmokkapa\AppData\Local\anaconda3\Lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:259: UserWarning: You tried to call render() but no `render_mode` was passed to the env constructor.
  warnings.warn("You tried to call render() but no `render_mode` was passed to the env constructor.")


(471.7, 56.62163897309932)

In [92]:
# Result= (468.3= reward, 64.66846217438605= std dev)
# for cartpole reward, it is calculated as 1 point for every step that the pole remains upright(with a max of 200 steps)
# If the pole is more than 15 degree from vertical or the cart moves more than 2.4 units from center, the episode ends.
env.close()

# 5. Testing

In [ ]:
# To test, we gonna take our observations pass into our agent. 
# Our agent try to determine the best type of action to take to our env and maximize the reward.

In [93]:
episodes = 5
for episode in range(1, episodes + 1):
    obs = env.reset()
    done = False
    score = 0
    
    while not done:
        env.render()  # This shows the environment window if using render_mode="human"  # render func allows us to view the the graphical representation of the env.
        action, _ = model.predict(obs)  #Now we are using our saved model.
        obs, reward, done, info = env.step(action)
        score += reward
    print("Episodes:{} score:{}".format(episode, score))
env.close()

Episodes:1 score:[294.]
Episodes:2 score:[287.]
Episodes:3 score:[209.]
Episodes:4 score:[326.]
Episodes:5 score:[119.]


In [100]:
obs = env.reset()
obs

array([[ 0.04929203,  0.03110939, -0.02446332, -0.02219671]],
      dtype=float32)

In [101]:
action, _ = model.predict(obs)

In [102]:
env.step(action)

(array([[ 0.04991421,  0.22657347, -0.02490725, -0.3224966 ]],
       dtype=float32),
 array([1.], dtype=float32),
 array([False]),
 [{'TimeLimit.truncated': False}])

# 6. Viewing Logs in Tensorboard

In [103]:
# we were able to train the model, get it up and running. 
# If you re training in a way more sophisticated env , we might wanna view the training logs inside the tensorboard.

training_log_path = os.path.join(log_path, "PPO_2")
training_log_path

'Training\\Logs\\PPO_2'

In [ ]:
# !tensorboard --logdir={training_log_path}
# using an ! mark inside the notebook is known as a magic cmd.

### 7. Applying Callbacks to training stage

In [104]:
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold

In [105]:
save_path = os.path.join("Training", "Saved Models")

In [106]:
stop_callback= StopTrainingOnRewardThreshold(reward_threshold= 200, verbose=1)
eval_callback = EvalCallback(env,
                             callback_on_new_best= stop_callback,
                             eval_freq= 10000,
                             best_model_save_path = save_path,
                             verbose= 1)


In [117]:
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=log_path)

Using cpu device


In [108]:
model.learn(total_timesteps= 20000, callback=eval_callback)

Logging to Training\Logs\PPO_11
-----------------------------
| time/              |      |
|    fps             | 1742 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1275        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009361366 |
|    clip_fraction        | 0.0959      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.686      |
|    explained_variance   | -0.0255     |
|    learning_rate        | 0.0003      |
|    loss                 | 9.09        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.014      |
|    value_loss           | 55.7        |
-----------------------------------------
--

c:\Users\nmokkapa\AppData\Local\anaconda3\Lib\site-packages\stable_baselines3\common\evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=10000, episode_reward=500.00 +/- 0.00
Episode length: 500.00 +/- 0.00
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 500          |
|    mean_reward          | 500          |
| time/                   |              |
|    total_timesteps      | 10000        |
| train/                  |              |
|    approx_kl            | 0.0071282443 |
|    clip_fraction        | 0.0541       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.612       |
|    explained_variance   | 0.284        |
|    learning_rate        | 0.0003       |
|    loss                 | 24.6         |
|    n_updates            | 40           |
|    policy_gradient_loss | -0.0138      |
|    value_loss           | 61.2         |
------------------------------------------
New best mean reward!
Stopping training because the mean reward 500.00  is above the threshold 200


## 8. Change Policies
* We can use a diff neural network architecture or we can use a different algorithm as well

In [110]:
net_arch = [dict(pi = [128,128,128,128], vf = [128,128,128,128])]
# we have designed a new neural network architecture and that we need to assign in our algorithm.

In [112]:
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=log_path, policy_kwargs= {'net_arch': net_arch})

Using cpu device


c:\Users\nmokkapa\AppData\Local\anaconda3\Lib\site-packages\stable_baselines3\common\policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


In [113]:
model.learn(total_timesteps= 20000, callback=eval_callback)

Logging to Training\Logs\PPO_12
-----------------------------
| time/              |      |
|    fps             | 1384 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 935         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013825458 |
|    clip_fraction        | 0.187       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.682      |
|    explained_variance   | -0.00375    |
|    learning_rate        | 0.0003      |
|    loss                 | 3.84        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0225     |
|    value_loss           | 20.4        |
-----------------------------------------
--

c:\Users\nmokkapa\AppData\Local\anaconda3\Lib\site-packages\stable_baselines3\common\evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=10000, episode_reward=491.60 +/- 16.80
Episode length: 491.60 +/- 16.80
-----------------------------------------
| eval/                   |             |
|    mean_ep_length       | 492         |
|    mean_reward          | 492         |
| time/                   |             |
|    total_timesteps      | 10000       |
| train/                  |             |
|    approx_kl            | 0.007507993 |
|    clip_fraction        | 0.0904      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.578      |
|    explained_variance   | 0.4         |
|    learning_rate        | 0.0003      |
|    loss                 | 15.7        |
|    n_updates            | 40          |
|    policy_gradient_loss | -0.017      |
|    value_loss           | 49.4        |
-----------------------------------------
------------------------------
| time/              |       |
|    fps             | 686   |
|    iterations      | 5     |
|    time_elapsed    | 14    

### 9. Using a different algorithm rather PPO

In [114]:
from stable_baselines3 import DQN

In [115]:
model = DQN("MlpPolicy", env, verbose=1, tensorboard_log=log_path)

Using cpu device


In [116]:
model.learn(total_timesteps= 20000, callback=eval_callback)

Logging to Training\Logs\DQN_1
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.954    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 7208     |
|    time_elapsed     | 0        |
|    total_timesteps  | 96       |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.92     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1648     |
|    time_elapsed     | 0        |
|    total_timesteps  | 169      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.492    |
|    n_updates        | 17       |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.862    |
| time/               |          |
|    episodes         | 12       |
|    fps              | 

c:\Users\nmokkapa\AppData\Local\anaconda3\Lib\site-packages\stable_baselines3\common\evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


----------------------------------
| rollout/            |          |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 932      |
|    fps              | 1143     |
|    time_elapsed     | 8        |
|    total_timesteps  | 9771     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00347  |
|    n_updates        | 2417     |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 936      |
|    fps              | 1142     |
|    time_elapsed     | 8        |
|    total_timesteps  | 9806     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0042   |
|    n_updates        | 2426     |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rat

In [ ]:
DQN.load